# k-Nearest Neighbors From Scratch

**Topic:** k-Nearest Neighbors &nbsp;·&nbsp; **Difficulty:** Medium

## The data (created in the next cell)

- `X_train` `(1200, 5)`, `y_train` `(1200,)` (3 classes). Feature 0 has a scale of hundreds but almost no class signal, and features 1–4 carry the signal at scale ~1.
- `X_test` `(400, 5)`: predict these. `y_test` is used by the evaluation cell.
- Raw-distance k-NN falls well short of the target here. Standardized k-NN does very well.

## Your task

With **NumPy only** (no `sklearn.neighbors`):

1. Standardize features using the **training** mean and standard deviation, and apply the same transform to the test set.
2. For each test point find the k nearest training points (Euclidean distance, k around 10) and take a majority vote.
3. Set `predictions` to the voted labels.

## Evaluation

Test accuracy must reach the target shown in the evaluation cell.


In [ ]:
import numpy as np

rng = np.random.default_rng(51)
centers = rng.normal(0, 2.2, (3, 4))     # signal lives in 4 dims

def make(n, seed):
    r = np.random.default_rng(seed)
    y = r.integers(0, 3, n)
    signal = centers[y] + r.normal(0, 0.8, (n, 4))
    loud_noise = r.normal(500, 120, (n, 1))   # huge scale, no signal
    X = np.hstack([loud_noise, signal])
    return X, y

X_train, y_train = make(1200, 61)
X_test, y_test = make(400, 62)
print(f"feature std devs: {X_train.std(axis=0).round(1)}")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: run this cell to compare your predictions to the truth
# =====================================================================
import numpy as np

TARGET_ACCURACY = 0.85

if "predictions" not in dir():
    raise NameError("Define `predictions` in the solution cell before running this cell.")

_pred = np.asarray(predictions).ravel()
_true = np.asarray(y_test).ravel()
if _pred.shape != _true.shape:
    raise ValueError(f"`predictions` has shape {_pred.shape}, expected {_true.shape}.")

accuracy = float((_pred == _true).mean())
print("=" * 52)
print(f"Test accuracy : {accuracy:.4f}")
print(f"Target        : >= {TARGET_ACCURACY:.2f}")
print("-" * 52)
print("Per-class recall:")
for _cls in np.unique(_true):
    _mask = _true == _cls
    print(f"  class {_cls!r:>6} : {float((_pred[_mask] == _cls).mean()):.3f}   ({int(_mask.sum())} test samples)")
print("=" * 52)
if accuracy >= TARGET_ACCURACY:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet, revisit your model and try again.")


## Solution

Standardize with the training mean/std (so the loud no-signal feature stops dominating the metric), compute all test-train distances with broadcasting, then majority-vote the k = 10 nearest labels.

```python
import numpy as np

mu, sd = X_train.mean(axis=0), X_train.std(axis=0)
Ztr = (X_train - mu) / sd
Zte = (X_test - mu) / sd

k = 10
dists = np.linalg.norm(Zte[:, None, :] - Ztr[None, :, :], axis=2)
nearest = np.argsort(dists, axis=1)[:, :k]
votes = y_train[nearest]
predictions = np.array([np.bincount(v.astype(np.intp), minlength=3).argmax() for v in votes])
```
